# Do Stricter Firearm Control Laws Reduce Firearm Mortality?

## Introduction

Firearm mortality is a significant public health concern in the United States, with substantial differences in death rates across states. Firearm regulations also vary considerably, raising questions about whether stricter laws are associated with lower firearm mortality.

The purpose of this analysis is to investigate the relationship between state firearm control laws and firearm mortality rates. Using data from the Centers for Disease Control and Prevention (CDC) and a five point scale of firearm law strictness, this project will compare states through heat maps and statistical analysis.

**Research Question:** Do states with stricter firearm control laws tend to have lower firearm mortality rates?

**Libraries for Data Collection, Analysis, and Visualization**

These libraries handle API requests (requests), tabular data (pandas), numerical operations (numpy), interactive charts (plotly), and statistical testing (scipy).

In [36]:

import pandas as pd
import numpy as np
import requests
import plotly.express as px
from scipy.stats import spearmanr

print("Libraries imported successfully!")


Libraries imported successfully!


**Data Source: Firearm Mortality**

Firearm mortality data were obtained from the Centers for Disease Control and Prevention (CDC) through its publicly available API. The dataset, *Mapping Injury, Overdose, and Violence — State*, provides firearm-related mortality rates across U.S. states.

For this analysis, records were filtered to include firearm-related deaths (`FA_Deaths`) during 2024. Washington, D.C., was excluded to maintain a consistent comparison of the 50 states. The resulting dataset contains one firearm mortality rate per state, expressed as deaths per 100,000 residents.

**Source:** CDC Open Data

https://data.cdc.gov/Injury-Violence/Mapping-Injury-Overdose-and-Violence-State/fpsi-y8tj

**API:** https://data.cdc.gov/resource/fpsi-y8tj.json[link text](https://)

In [37]:

# CDC Open Data API
url = "https://data.cdc.gov/resource/fpsi-y8tj.json"

# Request records from the CDC
response = requests.get(url, params={"$limit": 5000}, timeout=30)
response.raise_for_status()

# Convert JSON into a pandas DataFrame
cdc_raw = pd.DataFrame(response.json())

print("Dataset dimensions:", cdc_raw.shape)
print("\nColumn names:")
print(cdc_raw.columns.tolist())

display(cdc_raw.head())


Dataset dimensions: (2142, 8)

Column names:
['geoid', 'name', 'intent', 'period', 'count_sup', 'rate', 'data_as_of', 'ttm_date_range']


,geoid,name,intent,period,count_sup,rate,data_as_of,ttm_date_range
0,04,Arizona,FA_Deaths,TTM,1347,18.10000000000000,2026-09-04T00:00:00.000,"May, 2025 to April, 2026"
1,04,Arizona,FA_Homicide,TTM,298,4.000000000000000,2026-09-04T00:00:00.000,"May, 2025 to April, 2026"
2,04,Arizona,FA_Suicide,TTM,973,13.10000000000000,2026-09-04T00:00:00.000,"May, 2025 to April, 2026"
3,05,Arkansas,All_Homicide,TTM,215,7.000000000000000,2026-09-04T00:00:00.000,"May, 2025 to April, 2026"
4,05,Arkansas,All_Suicide,TTM,549,17.90000000000000,2026-09-04T00:00:00.000,"May, 2025 to April, 2026"


The request retrieves JSON records from the CDC open-data API and converts them into a DataFrame. The CDC API request returned 2,142 records with 8 fields. This process makes it possible to filter the observations, inspect the variables, and prepare the state level mortality data without manually copying figures. Displaying the dimensions, column names, and sample records helps verify the source structure.

In [38]:

firearm_raw = cdc_raw[
    cdc_raw["intent"] == "FA_Deaths"
].copy()

print("Available periods:")
print(sorted(firearm_raw["period"].dropna().unique()))

print("\nSample firearm records:")
display(firearm_raw.head(10))


Available periods:
['2019', '2020', '2021', '2022', '2023', '2024', 'TTM']

Sample firearm records:


,geoid,name,intent,period,count_sup,rate,data_as_of,ttm_date_range
0,04,Arizona,FA_Deaths,TTM,1347,18.10000000000000,2026-09-04T00:00:00.000,"May, 2025 to April, 2026"
6,05,Arkansas,FA_Deaths,TTM,566,18.50000000000000,2026-09-04T00:00:00.000,"May, 2025 to April, 2026"
12,06,California,FA_Deaths,TTM,2623,6.700000000000000,2026-09-04T00:00:00.000,"May, 2025 to April, 2026"
18,08,Colorado,FA_Deaths,TTM,912,15.50000000000000,2026-09-04T00:00:00.000,"May, 2025 to April, 2026"
24,09,Connecticut,FA_Deaths,TTM,162,4.500000000000000,2026-09-04T00:00:00.000,"May, 2025 to April, 2026"
30,10,Delaware,FA_Deaths,TTM,118,11.40000000000000,2026-09-04T00:00:00.000,"May, 2025 to April, 2026"
36,11,District of Columbia,FA_Deaths,TTM,95,14.00000000000000,2026-09-04T00:00:00.000,"May, 2025 to April, 2026"
42,12,Florida,FA_Deaths,TTM,2842,12.60000000000000,2026-09-04T00:00:00.000,"May, 2025 to April, 2026"
49,13,Georgia,FA_Deaths,TTM,1837,16.70000000000000,2026-09-04T00:00:00.000,"May, 2025 to April, 2026"
55,15,Hawaii,FA_Deaths,TTM,55,3.800000000000000,2026-09-04T00:00:00.000,"May, 2025 to April, 2026"


Filtering intent within the CDC dataset to FA_Deaths isolates all firearm related deaths rather than firearm homicide or suicide separately. Listing the available periods confirms which years can be analyzed, which in this case is 2019-2024.

In [39]:

# Select the 2024 calendar-year records
firearm_2024 = firearm_raw[
    firearm_raw["period"].astype(str).str.startswith("2024")
].copy()

# Convert rates to numeric values
firearm_2024["rate"] = pd.to_numeric(
    firearm_2024["rate"], errors="coerce"
)

# Keep the state and firearm mortality rate
mortality = firearm_2024[
    ["name", "rate"]
].rename(columns={
    "name": "State",
    "rate": "Mortality_Rate"
})

# Exclude Washington, D.C. (50 states only)
mortality = mortality[
    mortality["State"] != "District of Columbia"
]

mortality = mortality.sort_values("State").reset_index(drop=True)

print("Number of records:", len(mortality))
print("Unique states:", mortality["State"].nunique())
print("Missing rates:", mortality["Mortality_Rate"].isna().sum())

display(mortality.head(10))


Number of records: 50
Unique states: 50
Missing rates: 0


,State,Mortality_Rate
0,Alabama,23.7
1,Alaska,24.8
2,Arizona,17.9
3,Arkansas,20.8
4,California,7.3
5,Colorado,16.4
6,Connecticut,6.2
7,Delaware,11.9
8,Florida,14.1
9,Georgia,18.0


This step selects 2024 records, converts mortality rates to numbers, and keeps one observation for each of the 50 states. The missing value and state count checks confirm the data are ready for mapping.

In [40]:

state_abbr = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ",
    "Arkansas": "AR", "California": "CA", "Colorado": "CO",
    "Connecticut": "CT", "Delaware": "DE", "Florida": "FL",
    "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID",
    "Illinois": "IL", "Indiana": "IN", "Iowa": "IA",
    "Kansas": "KS", "Kentucky": "KY", "Louisiana": "LA",
    "Maine": "ME", "Maryland": "MD", "Massachusetts": "MA",
    "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE",
    "Nevada": "NV", "New Hampshire": "NH", "New Jersey": "NJ",
    "New Mexico": "NM", "New York": "NY", "North Carolina": "NC",
    "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI",
    "South Carolina": "SC", "South Dakota": "SD",
    "Tennessee": "TN", "Texas": "TX", "Utah": "UT",
    "Vermont": "VT", "Virginia": "VA", "Washington": "WA",
    "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY"
}

mortality["State_Abbr"] = mortality["State"].map(state_abbr)

print("Missing state abbreviations:",
      mortality["State_Abbr"].isna().sum())

display(mortality.head())


Missing state abbreviations: 0


,State,Mortality_Rate,State_Abbr
0,Alabama,23.7,AL
1,Alaska,24.8,AK
2,Arizona,17.9,AZ
3,Arkansas,20.8,AR
4,California,7.3,CA


Plotly identifies U.S. states using postal abbreviations, so this dictionary maps each state name to their two letter name. The missing value check catches unmatched names.

In [41]:

fig = px.choropleth(
    mortality,
    locations="State_Abbr",
    locationmode="USA-states",
    color="Mortality_Rate",
    scope="usa",
    hover_name="State",
    hover_data={"Mortality_Rate": ":.1f", "State_Abbr": False},
    color_continuous_scale="Viridis",
    labels={"Mortality_Rate": "Deaths per 100,000"},
    title="Firearm Mortality Rates Across U.S. States (2024)"
)

fig.update_layout(
    title_x=0.5,
    geo=dict(showlakes=False),
    margin=dict(l=20, r=20, t=70, b=20),
    coloraxis_colorbar=dict(
        title="Deaths per<br>100,000"
    )
)

fig.show()


The 2024 CDC heat map highlights substantial differences in firearm mortality across states. Darker colors indicate lower reported rates and brighter colors indicate higher rates. Mississippi (27.6) and New Mexico (26.6) appear among the highest, while Hawaii (3.9) and Massachusetts (4.1) are among the lowest, measured as deaths per 100,000 residents. These geographic differences motivate the comparison with firearm law classifications, but the map alone cannot explain their causes.


In [42]:
print("2024 Firearm Mortality Summary")
print("-" * 35)

print("Mean:", round(mortality["Mortality_Rate"].mean(), 2))
print("Median:", round(mortality["Mortality_Rate"].median(), 2))
print("Minimum:", mortality["Mortality_Rate"].min())
print("Maximum:", mortality["Mortality_Rate"].max())
print("Standard deviation:",
      round(mortality["Mortality_Rate"].std(), 2))


2024 Firearm Mortality Summary
-----------------------------------
Mean: 14.95
Median: 14.6
Minimum: 3.9
Maximum: 27.6
Standard deviation: 5.91


The mean and median describe the typical state rate, while minimum, maximum, and standard deviation describe the spread. Each state contributes equally to these statistics. The average firearm mortality rate across the 50 states was 14.95 deaths per 100,000 residents, with a median of 14.6. Rates ranged from 3.9 to 27.6, with a standard deviation of 5.91, indicating substantial variation between states. This variation provides a basis for investigating whether differences in firearm-law strictness are associated with mortality.

In [43]:
# Identify the five states with the highest mortality rates
highest = mortality.nlargest(5, "Mortality_Rate")

# Identify the five states with the lowest mortality rates
lowest = mortality.nsmallest(5, "Mortality_Rate")

print("Five states with highest firearm mortality rates:")
display(highest[["State", "Mortality_Rate"]])

print("\nFive states with lowest firearm mortality rates:")
display(lowest[["State", "Mortality_Rate"]])


Five states with highest firearm mortality rates:


,State,Mortality_Rate
23,Mississippi,27.6
30,New Mexico,26.6
1,Alaska,24.8
49,Wyoming,24.5
0,Alabama,23.7



Five states with lowest firearm mortality rates:


,State,Mortality_Rate
10,Hawaii,3.9
20,Massachusetts,4.1
29,New Jersey,4.2
31,New York,4.6
38,Rhode Island,5.0


The five states with the highest firearm mortality rates were Mississippi (27.6), New Mexico (26.6), Alaska (24.8), Wyoming (24.5), and Alabama (23.7) deaths per 100,000 residents. In comparison, Hawaii (3.9), Massachusetts (4.1), New Jersey (4.2), New York (4.6), and Rhode Island (5.0) had the lowest rates. These differences demonstrate the geographic variation in firearm mortality and provide a starting point for comparing state firearm laws.

In [44]:
# Calculate the range of firearm mortality rates
mortality_range = (
    mortality["Mortality_Rate"].max()
    - mortality["Mortality_Rate"].min()
)

print(
    "Difference between highest and lowest state:",
    round(mortality_range, 2),
    "deaths per 100,000"
)

Difference between highest and lowest state: 23.7 deaths per 100,000


The difference between the highest and lowest state firearm mortality rates was 23.7 deaths per 100,000 residents. This substantial range shows that firearm mortality varies considerably across the country, suggesting that state level differences are worth investigating further.

### Results: Firearm Mortality Across U.S. States

The 2024 CDC data reveal considerable variation in firearm mortality rates across the United States. The average state firearm mortality rate was **14.95 deaths per 100,000 residents**, with a median of 14.6 and a standard deviation of 5.91.

Mississippi had the highest firearm mortality rate at 27.6 deaths per 100,000 residents, followed by New Mexico (26.6), Alaska (24.8), Wyoming (24.5), and Alabama (23.7).

In contrast, Hawaii had the lowest rate at 3.9 deaths per 100,000 residents, followed by Massachusetts (4.1), New Jersey (4.2), New York (4.6), and Rhode Island (5.0).

The difference between the highest and lowest state mortality rates was **23.7 deaths per 100,000 residents**, demonstrating substantial geographic variation.

These findings establish a baseline for investigating whether differences in firearm-law strictness are associated with firearm mortality rates. However, mortality data alone cannot establish whether firearm regulations explain these differences.

In [45]:
likert_labels = {
    1: "Very lax",
    2: "Lax",
    3: "Moderate",
    4: "Strict",
    5: "Very strict"
}

def classify_laws(rank):
    if 1 <= rank <= 10:
        return 5
    elif 11 <= rank <= 20:
        return 4
    elif 21 <= rank <= 30:
        return 3
    elif 31 <= rank <= 40:
        return 2
    elif 41 <= rank <= 50:
        return 1
    else:
        return None

The function converts the source’s state rankings into five ordered law strictness categories. Ranks 1–10 receive a score of 5 (Very strict), while ranks 41–50 receive 1 (Very lax). These are relative ranking groups created for the assignment, not an independently validated legal threshold or a survey based Likert measure.

In [46]:
# State gun-law strength scores transcribed from Everytown's published rankings
# (2026). Source: https://everytownresearch.org/rankings/

law_data = [
    ("California", 91), ("Illinois", 87),
    ("Massachusetts", 86.5), ("New York", 85),
    ("Connecticut", 83), ("Hawaii", 81.5),
    ("New Jersey", 80.5), ("Maryland", 78.5),
    ("Washington", 76), ("Delaware", 71),
    ("Colorado", 69.5), ("Oregon", 67),
    ("Rhode Island", 65), ("Minnesota", 55),
    ("Virginia", 49.5), ("New Mexico", 43.5),
    ("Vermont", 43), ("Pennsylvania", 40),
    ("Nevada", 37.5), ("Michigan", 37),
    ("Maine", 33), ("Wisconsin", 28.5),
    ("Florida", 25.5), ("North Carolina", 24.5),
    ("Nebraska", 23), ("Indiana", 18.5),
    ("West Virginia", 18.5), ("Alabama", 16),
    ("Tennessee", 15), ("Ohio", 14.5),
    ("Iowa", 14), ("Texas", 14),
    ("South Carolina", 14), ("Louisiana", 12.5),
    ("North Dakota", 11.5), ("Utah", 11.5),
    ("Kentucky", 9.5), ("Kansas", 9.5),
    ("Alaska", 9), ("Missouri", 8),
    ("Arizona", 8), ("Oklahoma", 7.5),
    ("New Hampshire", 7.5), ("Georgia", 5.5),
    ("Montana", 5), ("Wyoming", 5),
    ("Arkansas", 4.5), ("South Dakota", 4),
    ("Mississippi", 4), ("Idaho", 3.5)
]

laws = pd.DataFrame(
    law_data,
    columns=["State", "Law_Strength_Score"]
)

# Published ordering, including states with tied scores
laws["Published_Rank"] = range(1, len(laws) + 1)

print("States in law dataset:", len(laws))
print("Duplicate states:", laws["State"].duplicated().sum())

display(laws.head())

States in law dataset: 50
Duplicate states: 0


,State,Law_Strength_Score,Published_Rank
0,California,91.0,1
1,Illinois,87.0,2
2,Massachusetts,86.5,3
3,New York,85.0,4
4,Connecticut,83.0,5


The Everytown dataset contains firearm-law strength scores for all 50 states, with California receiving the highest score of 91 and Idaho receiving the lowest score of 3.5. These scores provide a standardized measure of firearm law strength that can be compared with CDC firearm mortality rates. The published rankings are also retained to classify states into five equal sized groups.

In [47]:
laws["Law_Strictness"] = laws["Published_Rank"].apply(
    classify_laws
)

laws["Likert_Category"] = laws["Law_Strictness"].map(
    likert_labels
)

print("Number of states per category:")
display(
    laws.groupby(
        ["Law_Strictness", "Likert_Category"]
    ).size().reset_index(name="State_Count")
)

Number of states per category:


,Law_Strictness,Likert_Category,State_Count
0,1,Very lax,10
1,2,Lax,10
2,3,Moderate,10
3,4,Strict,10
4,5,Very strict,10


In [48]:
combined = mortality.merge(
    laws,
    on="State",
    how="inner",
    validate="one_to_one"
)

print("Total states:", len(combined))
print("\nMissing values:")
print(combined.isna().sum())

display(combined.head(10))

Total states: 50

Missing values:
State                 0
Mortality_Rate        0
State_Abbr            0
Law_Strength_Score    0
Published_Rank        0
Law_Strictness        0
Likert_Category       0
dtype: int64


,State,Mortality_Rate,State_Abbr,Law_Strength_Score,Published_Rank,Law_Strictness,Likert_Category
0,Alabama,23.7,AL,16.0,28,3,Moderate
1,Alaska,24.8,AK,9.0,39,2,Lax
2,Arizona,17.9,AZ,8.0,41,1,Very lax
3,Arkansas,20.8,AR,4.5,47,1,Very lax
4,California,7.3,CA,91.0,1,5,Very strict
5,Colorado,16.4,CO,69.5,11,4,Strict
6,Connecticut,6.2,CT,83.0,5,5,Very strict
7,Delaware,11.9,DE,71.0,10,5,Very strict
8,Florida,14.1,FL,25.5,23,3,Moderate
9,Georgia,18.0,GA,5.5,44,1,Very lax


### Firearm Law Strictness Classification

State firearm law strength scores were obtained from Everytown for Gun Safety's published 2026 gun-law rankings. These scores reflect Everytown's assessment of firearm policies across states.

For this exploratory analysis, states were divided into five equal sized groups based on their published rankings. Each group contains ten states, with a score of 1 representing the least restrictive group and 5 representing the most restrictive group.

The resulting five-point scale is an ordinal classification developed for this project, rather than a survey-based Likert scale.

**Methodological limitation:** The firearm mortality data represent 2024, while the firearm-law rankings represent 2026. Consequently, the analysis describes an association between measures from different years and cannot establish a causal effect of firearm regulations on mortality.

**Source:** https://everytownresearch.org/rankings/

In [49]:
fig2 = px.choropleth(
    combined,
    locations="State_Abbr",
    locationmode="USA-states",
    color="Likert_Category",
    category_orders={"Likert_Category": ["Very lax", "Lax", "Moderate", "Strict", "Very strict"]},
    color_discrete_map={"Very lax": "#440154", "Lax": "#3B528B", "Moderate": "#21918C", "Strict": "#5EC962", "Very strict": "#FDE725"},
    scope="usa",
    hover_name="State",
    hover_data={
        "Law_Strictness": True,
        "Likert_Category": True,
        "Law_Strength_Score": True,
        "State_Abbr": False
    },
    labels={"Law_Strictness": "Law Strictness (1–5)"},
    title="State Firearm-Law Strictness (2026)"
)

fig2.update_layout(
    title_x=0.5,
    margin=dict(l=20, r=20, t=70, b=20),
    legend_title_text="Law strictness (1–5)"
)

fig2.show()

This geographic heat map shows how the 2026 lawstrength categories vary by state, using five distinct, accessible colors. This map can be compared to the 2024 mortality map to look for broad geographic patterns, while remembering that neighboring states and other social factors may also influence mortality. The map shows a classification, not the number of laws or their individual effects.

In [50]:
category_summary = (
    combined.groupby(
        ["Law_Strictness", "Likert_Category"],
        as_index=False
    )
    .agg(
        Mean_Mortality=("Mortality_Rate", "mean"),
        Median_Mortality=("Mortality_Rate", "median"),
        State_Count=("State", "count")
    )
    .sort_values("Law_Strictness")
)

category_summary["Mean_Mortality"] = (
    category_summary["Mean_Mortality"].round(2)
)

category_summary["Median_Mortality"] = (
    category_summary["Median_Mortality"].round(2)
)

display(category_summary)

,Law_Strictness,Likert_Category,Mean_Mortality,Median_Mortality,State_Count
0,1,Very lax,19.51,18.80,10
1,2,Lax,17.33,17.20,10
2,3,Moderate,15.95,15.35,10
3,4,Strict,14.16,12.85,10
4,5,Very strict,7.80,6.75,10


Grouping states by law category shows average firearm mortality declining from 19.51 (Very lax) to 17.33 (Lax), 15.95 (Moderate), 14.16 (Strict), and 7.80 (Very strict) deaths per 100,000. Each category contains 10 states. The group means describe an association across states rather than an effect caused by legislation.

In [51]:
heatmap_data = category_summary.set_index(
    "Likert_Category"
)[["Mean_Mortality"]]


fig3 = px.imshow(
    heatmap_data,
    text_auto=".2f",
    color_continuous_scale="Viridis",
    aspect="auto",
    labels={
        "x": "",
        "y": "Law strictness",
        "color": "Deaths per 100,000"
    },
    title="Average Firearm Mortality by Law Strictness"
)

fig3.update_layout(
    title_x=0.5,
    width=900,
    height=550,
    margin=dict(l=120, r=130, t=90, b=70),
    xaxis=dict(
        title="",
        showticklabels=False
    ),
    yaxis_title="2026 law strictness category",
    coloraxis_colorbar=dict(
        title=dict(
            text="Deaths per 100,000",
            side="right"
        ),
        thickness=20,
        len=0.85
    )
)

fig3.show()

The heat map shows a consistent decrease in average firearm mortality as firearm law strictness increases. States with very lax laws had the highest average mortality rate at 19.51 deaths per 100,000 residents, followed by lax (17.33), moderate (15.95), and strict (14.16) states. States with very strict laws had the lowest average rate at 7.80 deaths per 100,000. The difference between the two extreme categories was 11.71 deaths per 100,000, suggesting a substantial negative association between firearm law strictness and mortality. However, this relationship does not establish causation.


In [52]:
correlation, p_value = spearmanr(
    combined["Law_Strictness"],
    combined["Mortality_Rate"]
)

print(f"Spearman correlation: {correlation:.3f}")
print(f"P-value: {p_value:.8g}")

Spearman correlation: -0.651
P-value: 3.1178307e-07


Spearman's correlation was ρ = −0.651, indicating a moderately strong negative association between firearm-law strictness and firearm mortality rates. The result was statistically significant (p < 0.001), suggesting that states with stricter firearm laws generally had lower mortality rates. However, correlation does not establish causation, and this analysis does not account for other factors that may influence firearm mortality.

### Results: Firearm-Law Strictness and Mortality

The comparison revealed a consistent negative association between firearm-law strictness and firearm mortality rates across the five categories.

States in the least restrictive category had an average firearm mortality rate of 19.51 deaths per 100,000 residents, compared with 7.80 deaths per 100,000 in the most restrictive category. The three intermediate categories also showed progressively lower average mortality rates as firearm-law strictness increased.

These findings suggest that states classified as having stricter firearm regulations tend to experience lower firearm mortality rates. However, this analysis is observational and cannot establish a causal relationship.

Other factors, including demographic differences, socioeconomic conditions, firearm ownership patterns, and differences in suicide and homicide rates, may contribute to the observed relationship.

Additionally, the mortality data are from 2024, whereas the law strength classifications are from 2026. This difference in measurement years limits the conclusions that can be drawn.

## Conclusion

This analysis investigated the relationship between state firearm law strictness and firearm mortality rates across the 50 U.S. states.

The findings revealed a negative association between firearm law strictness and firearm mortality. States in the least restrictive category had an average mortality rate of 19.51 deaths per 100,000 residents, compared with 7.80 in the most restrictive category.

Spearman's rank correlation coefficient was **−0.651 (p < 0.001)**, indicating a statistically significant negative association between firearm law strictness and mortality rates.

The geographic heat maps illustrated substantial differences in firearm mortality and firearm law classifications across states. The category comparison further demonstrated that average firearm mortality decreased across increasingly restrictive law categories.

However, these findings do not establish causation. The analysis does not control for demographic, socioeconomic, geographic, or firearm ownership differences between states. Furthermore, the mortality data represent 2024, while the law strength classifications represent 2026.

Overall, the results suggest that states classified as having stricter firearm regulations tend to have lower firearm mortality rates, although additional research using temporally aligned data and controls for potential confounding variables is necessary to evaluate causal effects.